# LOC Anomaly Detection — Tiered Pipeline

Reads pre-aggregated rate tables built by `loc_agg.sql`, then runs a two-tier
detection pipeline for each population (M&R FFS, C&S DSNP, OAH):

1. **Tier 1 — Percentile flagging**: flags entities in the top/bottom 10th percentile of each rate feature within their dimension. Directly actionable, no model needed.
2. **Tier 2 — Isolation Forest**: catches entities with an unusual *combination* of rates that no single KPI flag would surface.

**Run order:**
1. Run `loc_agg.sql` in Snowflake (creates the 3 `kn_loc_*_agg` tables)
2. Run this notebook (reads those tables, tiers 1 & 2, exports CSVs)

## Step 0: Setup & Configuration

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

warnings.filterwarnings("ignore", category = RuntimeWarning)

In [15]:
# --- Update each run (must match loc_agg.sql) ---
NOTIFICATIONS_DATE = "04222026"
OUTPUT_DATE        = "202604"

OUTPUT_DIR = r"C:\Users\knguy139\Documents\Projects\Data\Output"
os.makedirs(OUTPUT_DIR, exist_ok = True)

# Table name -> output file prefix, one entry per population
POPULATIONS = [
    (f"tmp_1m.kn_loc_mnr_agg_{NOTIFICATIONS_DATE}", "mnr"),
    (f"tmp_1m.kn_loc_cns_agg_{NOTIFICATIONS_DATE}", "cns"),
    (f"tmp_1m.kn_loc_oah_agg_{NOTIFICATIONS_DATE}", "oah"),
]

# --- Tier 1: percentile flagging ---
TOP_PCT    = 0.10   # flag entities at or above the 90th percentile
BOTTOM_PCT = 0.10   # flag entities at or below the 10th percentile

# --- Tier 2: Isolation Forest ---
CONTAMINATION = 0.05   # expected share of anomalies -- business decision, not statistical
TOP_N         = 20     # narratives written per dimension

# --- Rate features (column names from loc_agg.sql) ---
RATE_FEATURES = [
    "adr_rate",
    "persistent_adr_rate",
    "persistency",
    "md_review_rate",
    "appeal_rate",
    "appeal_overturn_rate",
    "p2p_rate",
    "p2p_overturn_rate",
    "mcr_reconsideration_rate",
    "mcr_overturn_rate",
    "member_appeal_rate",
    "member_appeal_overturn_rate",
    "pre_auth_rate",
    "auth_per_k",
]

# --- Labels and formats for plain-English narratives ---
FEATURE_META = {
    "adr_rate":                    ("ADR rate",                                "{:.1%}"),
    "persistent_adr_rate":         ("Persistent ADR rate",                     "{:.1%}"),
    "persistency":                 ("Persistency (persistent / initial ADR)",  "{:.1%}"),
    "md_review_rate":              ("MD review rate",                          "{:.1%}"),
    "appeal_rate":                 ("Appeal rate (% of ADRs)",                 "{:.1%}"),
    "appeal_overturn_rate":        ("Appeal overturn rate (% of ADRs)",        "{:.1%}"),
    "p2p_rate":                    ("P2P rate (% of ADRs)",                    "{:.1%}"),
    "p2p_overturn_rate":           ("P2P overturn rate (% of ADRs)",           "{:.1%}"),
    "mcr_reconsideration_rate":    ("MCR reconsideration rate (% of ADRs)",    "{:.1%}"),
    "mcr_overturn_rate":           ("MCR overturn rate (% of ADRs)",           "{:.1%}"),
    "member_appeal_rate":          ("Member appeal rate (% of ADRs)",          "{:.1%}"),
    "member_appeal_overturn_rate": ("Member appeal overturn rate (% of ADRs)", "{:.1%}"),
    "pre_auth_rate":               ("Pre-auth rate",                           "{:.1%}"),
    "auth_per_k":                  ("Auth per 1,000 members",                  "{:.1f}"),
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Populations: {[p[1].upper() for p in POPULATIONS]}")

Output directory: C:\Users\knguy139\Documents\Projects\Data\Output
Populations: ['MNR', 'CNS', 'OAH']


## Step 1: Connect to Snowflake & Load Data

In [5]:
engine = create_engine(
    URL(
        account = "UHG-UHGDWAAS",
        user = "KHANG.NGUYEN@UHC.COM",
        authenticator = "externalbrowser",
        role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
        warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
        database = "VING_PRD_TREND_DB",
        schema = "TMP_1M"
    )
)

In [6]:
def load_aggregated(engine, table):
    """Read a pre-aggregated population table from Snowflake."""
    with engine.connect() as conn:
        return (
            pd.read_sql(f"select * from {table}", conn)
            .rename(columns = str.lower)
        )

## Step 2: Tier 1 — Percentile Flagging

For each dimension x rate feature, rank every entity against its peers.  
Flag top/bottom 10th percentile. No model — just descriptive statistics.  
This is the primary output: directly answers "who has a high overturn rate?"

In [7]:
def percentile_flags(df, rate_features, top_pct = TOP_PCT, bottom_pct = BOTTOM_PCT):
    """Rank entities on each rate feature within each dimension. Flag outliers.

    Returns long-format table: one row per entity x feature x dimension.
    """
    rows = []
    for dim, group in df.groupby("_dimension"):
        for feat in rate_features:
            if feat not in group.columns:
                continue
            ranked = group[["_dim_value", "case_count", feat]].dropna(subset = [feat]).copy()
            if ranked.empty:
                continue
            ranked["percentile"]  = ranked[feat].rank(pct = True)
            ranked["peer_median"] = ranked[feat].median()
            ranked["flag_high"]   = ranked["percentile"] >= (1 - top_pct)
            ranked["flag_low"]    = ranked["percentile"] <= bottom_pct
            ranked["_dimension"]  = dim
            ranked["_feature"]    = feat
            rows.append(ranked)
    return pd.concat(rows, ignore_index = True) if rows else pd.DataFrame()

## Step 3: Tier 2 — Isolation Forest

Secondary sweep for multivariate anomalies.  
Catches entities whose *combination* of rates is unusual even when no single rate
is extreme enough to trigger a Tier 1 percentile flag.  
Features are scaled first so no single rate dominates the model.

In [8]:
def run_model(df, label):
    """Fit Isolation Forest on available rate features within one dimension."""
    available = [f for f in RATE_FEATURES if f in df.columns and df[f].notna().any()]
    if not available:
        return df

    X = df[available].fillna(df[available].median())
    X_scaled = StandardScaler().fit_transform(X)

    model = IsolationForest(
        n_estimators  = 200,
        contamination = CONTAMINATION,
        random_state  = 42,
        n_jobs        = -1,
    )
    model.fit(X_scaled)

    flagged = (model.decision_function(X_scaled) < 0).sum()
    print(f"  [{label}]  n = {len(df):,}  |  flagged = {flagged:,}")

    return df.assign(
        raw_score      = model.decision_function(X_scaled),
        _features_used = ",".join(available),
    )

## Step 4: Z-Scores & Top Reasons (Tier 2 interpretability)

Isolation Forest is a black box. Z-scores add interpretability after the fact:  
which features are most extreme relative to the dimension's mean?

In [9]:
def add_z_scores(df):
    """Append z-score columns for each rate feature."""
    z_cols = {
        f"{feat}_z": (df[feat] - df[feat].mean()) / df[feat].std()
        for feat in RATE_FEATURES
        if feat in df.columns and df[feat].std() > 0
    }
    return df.assign(**z_cols)


def top_reasons(row):
    """Identify the top 3 most extreme z-score features for one row."""
    z_vals = {
        feat: abs(row[f"{feat}_z"])
        for feat in RATE_FEATURES
        if f"{feat}_z" in row.index and pd.notna(row[f"{feat}_z"])
    }
    top3 = sorted(z_vals, key = z_vals.get, reverse = True)[:3]

    reasons = {}
    for i, feat in enumerate(top3, start = 1):
        z         = row[f"{feat}_z"]
        direction = "above" if z > 0 else "below"
        reasons[f"top_reason_{i}"] = f"{feat} ({z:+.1f}\u03c3 {direction} peer mean)"
    for i in range(len(top3) + 1, 4):
        reasons[f"top_reason_{i}"] = ""

    return pd.Series(reasons)

## Step 5: Plain-English Narratives (Tier 2)

For the top 20 most anomalous entities per dimension, generate a human-readable  
paragraph explaining what is unusual about their pattern relative to peers.

In [10]:
def generate_narrative(row, peer_medians, dim):
    """Build a plain-English narrative for one anomalous entity."""
    lines = [
        f"{dim} = {row['_dim_value']}  |  "
        f"cases = {int(row['case_count']):,}  |  "
        f"anomaly score = {row['raw_score']:.3f}"
    ]

    for feat, (label, fmt) in FEATURE_META.items():
        z_col = f"{feat}_z"
        if feat not in row.index or pd.isna(row.get(feat)):
            continue
        z = row.get(z_col, np.nan)
        if pd.isna(z) or abs(z) < 1.5:
            continue
        direction = "above" if z > 0 else "below"
        lines.append(
            f"  \u2022 {label}: {fmt.format(row[feat])} "
            f"vs. peer median {fmt.format(peer_medians.get(feat, np.nan))} "
            f"({z:+.1f}\u03c3 {direction})"
        )

    if len(lines) == 1:
        lines.append("  \u2022 No individual feature stands out; flagged on combined pattern.")

    return "\n".join(lines)

## Step 6: Combined Flagging

Merge Tier 1 (percentile) and Tier 2 (IF) flags into a single entity-level table  
with a `tier` column: `"percentile"`, `"isolation_forest"`, or `"both"`.

In [11]:
def combine_flags(pctl_df, if_scored):
    """Merge Tier 1 and Tier 2 flags into a single entity-level summary.

    Returns wide-format: one row per (_dimension, _dim_value) with tier column.
    """
    # Tier 1: entities flagged on any feature
    t1_flagged = set()
    if not pctl_df.empty:
        flagged_rows = pctl_df[pctl_df["flag_high"] | pctl_df["flag_low"]]
        t1_flagged = set(
            flagged_rows[["_dimension", "_dim_value"]]
            .apply(tuple, axis = 1)
        )

    # Tier 2: entities with negative raw_score
    t2_flagged = set()
    if "raw_score" in if_scored.columns:
        neg = if_scored[if_scored["raw_score"] < 0]
        t2_flagged = set(
            neg[["_dimension", "_dim_value"]]
            .apply(tuple, axis = 1)
        )

    # Assign tier label on the IF scored table (wide format, one row per entity)
    def assign_tier(row):
        key = (row["_dimension"], row["_dim_value"])
        in_t1 = key in t1_flagged
        in_t2 = key in t2_flagged
        if in_t1 and in_t2:
            return "both"
        elif in_t1:
            return "percentile"
        elif in_t2:
            return "isolation_forest"
        return ""

    return if_scored.assign(tier = if_scored.apply(assign_tier, axis = 1))

## Step 7: Run All Populations

For each population:
1. Load the pre-aggregated table
2. **Tier 1**: percentile flagging per dimension x feature
3. **Tier 2**: Isolation Forest per dimension, z-scores, top reasons
4. Combine flags and export CSVs

In [13]:
def run_population(engine, table, prefix):
    """Full tiered pipeline for one population."""
    pctl_csv      = os.path.join(OUTPUT_DIR, f"loc_pctl_{prefix}_{OUTPUT_DATE}.csv")
    if_csv        = os.path.join(OUTPUT_DIR, f"loc_if_{prefix}_{OUTPUT_DATE}.csv")
    narrative_csv = os.path.join(OUTPUT_DIR, f"loc_narratives_{prefix}_{OUTPUT_DATE}.csv")
    combined_csv  = os.path.join(OUTPUT_DIR, f"loc_combined_{prefix}_{OUTPUT_DATE}.csv")

    print(f"\n{'=' * 60}")
    print(f"  Population: {prefix.upper()}  |  {table}")

    df         = load_aggregated(engine, table)
    dimensions = df["_dimension"].unique()
    print(f"  {len(df):,} rows  |  dimensions: {list(dimensions)}")

    # ---- Tier 1: Percentile flagging ----
    print(f"\n  --- Tier 1: Percentile Flagging ---")
    pctl = percentile_flags(df, RATE_FEATURES)
    if not pctl.empty:
        n_flagged = pctl["flag_high"].sum() + pctl["flag_low"].sum()
        n_entities = pctl.loc[pctl["flag_high"] | pctl["flag_low"], "_dim_value"].nunique()
        print(f"  {n_flagged:,} flags across {n_entities:,} distinct entities")
        pctl.to_csv(pctl_csv, index = False)
        print(f"  Tier 1 output -> {pctl_csv}")
    else:
        print(f"  No Tier 1 flags.")

    # ---- Tier 2: Isolation Forest ----
    print(f"\n  --- Tier 2: Isolation Forest ---")
    all_scored = []
    for dim, group in df.groupby("_dimension"):
        scored = (
            group
            .pipe(run_model, label = dim)
            .pipe(add_z_scores)
        )
        scored[["top_reason_1", "top_reason_2", "top_reason_3"]] = scored.apply(
            top_reasons, axis = 1
        )
        all_scored.append(scored)

    if not all_scored:
        print(f"  No results for {prefix} -- skipping.")
        return None, None

    if_scored = (
        pd.concat(all_scored, ignore_index = True)
        .sort_values(["_dimension", "raw_score"], ascending = [True, True])
    )

    # Export Tier 2 scores
    id_cols    = ["_dimension", "_dim_value", "case_count", "membership"]
    rate_cols  = [f for f in RATE_FEATURES if f in if_scored.columns]
    score_cols = ["raw_score", "_features_used", "top_reason_1", "top_reason_2", "top_reason_3"]
    if_scored[[c for c in id_cols + rate_cols + score_cols if c in if_scored.columns]].to_csv(
        if_csv, index = False
    )
    print(f"  Tier 2 output -> {if_csv}")

    # Tier 2 narratives
    narrative_rows = []
    for dim, group in if_scored.groupby("_dimension"):
        top_n        = group.nsmallest(TOP_N, "raw_score")
        peer_medians = group[[f for f in RATE_FEATURES if f in group.columns]].median()
        for _, row in top_n.iterrows():
            narrative_rows.append({
                "dimension":  dim,
                "dim_value":  row["_dim_value"],
                "case_count": int(row["case_count"]),
                "raw_score":  row["raw_score"],
                "narrative":  generate_narrative(row, peer_medians, dim),
            })
    pd.DataFrame(narrative_rows).to_csv(narrative_csv, index = False)
    print(f"  Narratives    -> {narrative_csv}")

    # ---- Combined flags ----
    combined = combine_flags(pctl, if_scored)
    combined[[c for c in id_cols + rate_cols + ["raw_score", "tier"] if c in combined.columns]].to_csv(
        combined_csv, index = False
    )
    flagged_either = (combined["tier"] != "").sum()
    print(f"\n  Combined: {flagged_either:,} entities flagged by at least one tier")
    print(f"  Combined  -> {combined_csv}")

    return pctl, combined

In [16]:
results = {}
for table, prefix in POPULATIONS:
    results[prefix] = run_population(engine, table, prefix)

print(f"\n{'=' * 60}")
print("Done.")


  Population: MNR  |  tmp_1m.kn_loc_mnr_agg_04222026
  23,512 rows  |  dimensions: ['prov_tin', 'hospital_group', 'global', 'fin_market', 'admit_type', 'ipa_li_split', 'los_categories', 'ip_type', 'svc_setting']

  --- Tier 1: Percentile Flagging ---
  45,324 flags across 16,815 distinct entities
  Tier 1 output -> C:\Users\knguy139\Documents\Projects\Data\Output\loc_pctl_mnr_202604.csv

  --- Tier 2: Isolation Forest ---
  [admit_type]  n = 5  |  flagged = 1
  [fin_market]  n = 54  |  flagged = 3
  [global]  n = 18,184  |  flagged = 910
  [hospital_group]  n = 1,656  |  flagged = 83
  [ip_type]  n = 3  |  flagged = 1
  [ipa_li_split]  n = 5  |  flagged = 1
  [los_categories]  n = 7  |  flagged = 1
  [prov_tin]  n = 3,597  |  flagged = 180
  [svc_setting]  n = 1  |  flagged = 0
  Tier 2 output -> C:\Users\knguy139\Documents\Projects\Data\Output\loc_if_mnr_202604.csv
  Narratives    -> C:\Users\knguy139\Documents\Projects\Data\Output\loc_narratives_mnr_202604.csv

  Combined: 16,821 en

## Step 8: Quick Summary

In [17]:
for prefix, (pctl, combined) in results.items():
    print(f"--- {prefix.upper()} ---")
    if pctl is None:
        print("  no data\n")
        continue

    # Tier 1
    t1_flags = pctl["flag_high"].sum() + pctl["flag_low"].sum()
    print(f"  Tier 1 (percentile):  {t1_flags:,} feature-level flags")

    # Tier 2
    t2_flagged = (combined["raw_score"] < 0).sum() if "raw_score" in combined.columns else 0
    print(f"  Tier 2 (IF):          {t2_flagged:,} entities flagged")

    # Combined
    for tier_val in ["percentile", "isolation_forest", "both"]:
        n = (combined["tier"] == tier_val).sum()
        if n > 0:
            print(f"    tier={tier_val}: {n:,}")
    print()

--- MNR ---
  Tier 1 (percentile):  45,324 feature-level flags
  Tier 2 (IF):          1,180 entities flagged
    tier=percentile: 15,641
    tier=both: 1,180

--- CNS ---
  Tier 1 (percentile):  16,472 feature-level flags
  Tier 2 (IF):          444 entities flagged
    tier=percentile: 5,660
    tier=both: 444

--- OAH ---
  Tier 1 (percentile):  17,134 feature-level flags
  Tier 2 (IF):          486 entities flagged
    tier=percentile: 5,998
    tier=isolation_forest: 3
    tier=both: 483



In [ ]:
engine.dispose()
print("Connection closed.")